In [33]:
%pip install polars

Note: you may need to restart the kernel to use updated packages.


In [85]:
import polars as pl
import os
from dataclasses import dataclass, field
from typing import Optional
import altair as alt
alt.data_transformers.enable("vegafusion") 
# import matplotlib.pyplot as plt

DataTransformerRegistry.enable('vegafusion')

In [166]:
os.chdir("/home/tsetsi/Python/financial-data-science-jupyter")
WORK_DIR = os.getcwd()
DATA = os.path.join(WORK_DIR, "data")

In [167]:
files = {
    "quotes_inc_eu": "DE0007500001_quotes_incremental.csv",
    "quotes_inc_us": "US2561631068_quotes_incremental.csv",
    "trades_eu": "DE0007500001_trades.csv",
    "trades_us": "US2561631068_trades.csv"
}

In [168]:
quotes_inc_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns"),
    "price": pl.Float64,
    "best_bid_price": pl.Float64,
    "best_ask_price": pl.Float64,
}

trades_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

quotes_inc_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

trades_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

In [169]:
@dataclass
class LimitOrderBook:
    file: str
    folder_path: str
    df: pl.LazyFrame = field(init=False)
    schema_override: Optional[dict] = None
    separator: str = ","

    def __post_init__(self):
        # pl.scan_csv doesn't load the file into memory immediately but only when called with the
        # .collect() method, which generally makes running the code significantly faster.
        # schema_overrides is optional and can be used to explicitly set a data type to a column,
        # but it will return an error if polars finds some kind of mismatch.
        # def get_data(file: str, separator: str = ",", schema_overrides: dict = None) -> pl.LazyFrame:
        self.df = pl.scan_csv(
            source=f"{self.folder_path}/{self.file}",
            separator=self.separator,
            schema_overrides=self.schema_override
        )

In [192]:
## EU: Incremental quotes
quotes_inc_eu = LimitOrderBook(
    file=files["quotes_inc_eu"],
    folder_path=DATA,
    schema_override=quotes_inc_eu_schema
)

In [193]:
quotes_inc_eu = (
    quotes_inc_eu.df.sort(by=[pl.col("original_order_id"), 
                              pl.col("event_timestamp")]
))


In [194]:
# Sanity check: No original_order_id's exist within more than one venue.
print(
    quotes_inc_eu.group_by("original_order_id") \
        .agg([
            pl.col("venue").unique().alias("venues")
        ])
        .filter(pl.col("venues").list.len() > 1)
        .collect()
)

shape: (0, 2)
┌───────────────────┬───────────┐
│ original_order_id ┆ venues    │
│ ---               ┆ ---       │
│ i64               ┆ list[str] │
╞═══════════════════╪═══════════╡
└───────────────────┴───────────┘


In [203]:
def show_histogram(df: pl.LazyFrame, column_name: str, x_label: str = None, y_label: str = None) -> None:
    
    x=alt.X(f"{column_name}:N", sort="-y", axis=alt.Axis(labelAngle=0, title=x_label or column_name))
    
    (
        alt.layer(
            alt.Chart(df.collect()).mark_bar().encode(
                x=x,
                y="count():Q",
            ),
            alt.Chart(df.collect()).mark_text(dy=-5).encode(
                x=x,
                y="count():Q",
                text="count():Q",
            ),
        )
        .properties(width=750, height=250)
    ).display()

In [207]:
show_histogram(
    df=quotes_inc_eu, 
    column_name="venue",
    x_label="Venues"
)
show_histogram(
    df=quotes_inc_eu, 
    column_name="market_state",
    x_label="Market states"
)
show_histogram(
    df=quotes_inc_eu, 
    column_name="lob_action",
    x_label="LOB Actions"
)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

In [182]:
# Sanity check: Every order should have a REMOVE operation at some point
# and at some market state
orders_without_remove = (
    quotes_inc_eu.group_by(pl.col("original_order_id"))
    .agg(pl.col("lob_action").unique().alias("lob_actions"))
    .filter(
        # pl.col("lob_actions").list.contains("INSERT") &
        pl.col("lob_actions").list.contains("REMOVE").not_()
    )
)

orders_without_remove.collect().show(limit=20)

original_order_id,lob_actions
i64,list[str]


# 3.1.

In [211]:
quotes_aggs = [
    pl.col("venue") \
        .first()
        .alias("venue"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_date"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("removal_date"),
    pl.col("event_timestamp") \
        .max()
        .alias("latest_event_timestamp"),
    pl.col("lob_action") \
        .eq("UPDATE")
        .sum()
        .alias("number_of_updates"),
    pl.col("price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("price_at_insertion"),
    pl.col("size") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("size_at_insertion"),
    # EXECUTION SIZE
    # This sums the execution sizes of all original_order_id's
    # in preparation for determining the removal mechanism.
    pl.col("execution_size") \
        .filter(pl.col("order_executed")==True)
        .sum() # .first()
        .alias("execution_size"),
    pl.col("price_level") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_level"),
    pl.col("best_bid_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("best_bid_price_at_insertion"),
    pl.col("best_ask_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("best_ask_price_at_insertion"),
    pl.struct(
        [
            pl.col("event_timestamp") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("lob_updates"),
            pl.col("price") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_prices"),
            pl.col("size") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_sizes"),
        ]),
    pl.col("market_state") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("market_state_at_removal"),
]

quotes_calc = [
    (
        pl.when(pl.col("removal_date").is_not_null())
            .then(pl.col("removal_date"))
            .otherwise(pl.col("latest_event_timestamp"))
        - pl.col("insertion_date")
    ).alias("order_lifetime"),

    # (best bid price + best ask price) / 2
    (
        (
            pl.col("best_bid_price_at_insertion")
            + pl.col("best_ask_price_at_insertion")
        ) / 2
    ).alias("midpoint_at_insertion"),
]

In [212]:
quotes_collapsed = (
    quotes_inc_eu
    .group_by("original_order_id")
    .agg(quotes_aggs)
    .with_columns(quotes_calc)
)

quotes_collapsed = quotes_collapsed.with_columns([
    # Absolute distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    (
        pl.col("price_at_insertion")
        .sub(pl.col("midpoint_at_insertion"))
        .abs()
        .alias("abs_distance_to_midpoint")
    ),
    # Relative distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    # / Midpoint at Insertion
    (
        (
            pl.col("price_at_insertion")
            .sub(pl.col("midpoint_at_insertion"))
            .abs()
        )
        .truediv(pl.col("midpoint_at_insertion"))
        .alias("rel_distance_to_midpoint")
    ),
])

In [220]:
# Sanity check: No order should have an order lifetime = 0,
# which would mess up the logarithm.
quotes_collapsed \
    .filter(pl.col("order_lifetime")==0) \
    .collect() \
    .show(limit=10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,market_state_at_removal,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,i64,i64,f64,f64,list[struct[3]],str,duration[ns],f64,f64,f64


In [214]:
quotes_inc_eu \
    .filter((pl.col("order_executed")==False)) \
    .collect() \
    .show(limit=10)

side,price,size,order_id,event_timestamp,lob_action,old_price,old_size,old_order_id,order_executed,execution_price,execution_size,price_level,old_price_level,best_ask_price,best_ask_size,total_ask_size,best_bid_price,best_bid_size,total_bid_size,is_new_best_price,is_new_best_size,original_order_id,trade_id,size_ahead,orders_ahead,best_ask_num_orders,best_bid_num_orders,level_num_orders_total,level_size_total,total_ask_orders,total_bid_orders,market_state,venue
str,f64,i64,i64,datetime[ns],str,f64,i64,i64,bool,f64,i64,i64,i64,f64,i64,i64,f64,i64,i64,bool,bool,i64,i128,i64,i64,i64,i64,i64,i64,i64,i64,str,str
"""ASK""",8.1,947,50137,2023-09-01 07:00:23.832485,"""INSERT""",null,0,null,false,null,0,1,0,8.1,947,947,null,0,0,true,true,50137,0,0,0,1,0,1,947,1,0,"""CONTINUOUS_TRADING""","""AQEU"""
"""ASK""",null,0,null,2023-09-01 11:00:00.100847,"""REMOVE""",8.1,947,50137,false,null,0,0,10,7.304,750,6691,7.272,596,3742,false,false,50137,0,6691,11,1,1,0,0,11,7,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",5.0,757,50138,2023-09-01 07:00:23.832985,"""INSERT""",null,0,null,false,null,0,1,0,8.1,947,947,5.0,757,757,true,true,50138,0,0,0,1,1,1,757,1,1,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 11:00:00.100871,"""REMOVE""",5.0,757,50138,false,null,0,0,7,7.304,750,6691,7.272,596,2985,false,false,50138,0,2985,6,1,1,0,0,11,6,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.104,750,51441,2023-09-01 07:00:24.633269,"""INSERT""",null,0,null,false,null,0,1,0,8.1,947,947,7.104,750,1507,true,true,51441,0,0,0,1,1,1,750,1,2,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:00:24.642583,"""REMOVE""",7.104,750,51441,false,null,0,1,1,8.1,947,947,7.104,750,1507,false,true,51441,0,0,0,1,1,1,750,1,2,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.102,750,51442,2023-09-01 07:00:24.633271,"""INSERT""",null,0,null,false,null,0,2,0,8.1,947,947,7.104,750,2257,false,false,51442,0,750,1,1,1,1,750,1,3,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:00:24.633286,"""REMOVE""",7.102,750,51442,false,null,0,0,2,8.1,947,947,7.104,1500,2257,false,false,51442,0,1500,2,1,2,0,0,1,3,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.104,750,51443,2023-09-01 07:00:24.633283,"""INSERT""",null,0,null,false,null,0,1,1,8.1,947,947,7.104,1500,3007,false,true,51443,0,750,1,1,2,2,1500,1,4,"""CONTINUOUS_TRADING""","""AQEU"""


In [221]:
test_ids = [272540, 1473130, 94755, 51443] # [1693551612541526740, 1693551530008208487, 1693551623820260384]

quotes_inc_eu \
    .filter(pl.col("original_order_id").is_in(test_ids)) \
    .sort(by=[pl.col("original_order_id"), pl.col("event_timestamp")]) \
    .collect() \
    .show(limit=10)

quotes_collapsed \
    .filter(pl.col("original_order_id").is_in(test_ids)) \
    .collect() \
    .show(limit=10)

side,price,size,order_id,event_timestamp,lob_action,old_price,old_size,old_order_id,order_executed,execution_price,execution_size,price_level,old_price_level,best_ask_price,best_ask_size,total_ask_size,best_bid_price,best_bid_size,total_bid_size,is_new_best_price,is_new_best_size,original_order_id,trade_id,size_ahead,orders_ahead,best_ask_num_orders,best_bid_num_orders,level_num_orders_total,level_size_total,total_ask_orders,total_bid_orders,market_state,venue
str,f64,i64,i64,datetime[ns],str,f64,i64,i64,bool,f64,i64,i64,i64,f64,i64,i64,f64,i64,i64,bool,bool,i64,i128,i64,i64,i64,i64,i64,i64,i64,i64,str,str
"""BID""",7.104,750,51443,2023-09-01 07:00:24.633283,"""INSERT""",null,0,null,false,null,0,1,1,8.1,947,947,7.104,1500,3007,false,true,51443,0,750,1,1,2,2,1500,1,4,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:00:24.642585,"""REMOVE""",7.104,750,51443,false,null,0,0,1,8.1,947,947,5.0,757,757,true,true,51443,0,0,0,1,1,0,0,1,1,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.12,138,94755,2023-09-01 07:01:06.903164,"""INSERT""",null,0,null,false,null,0,1,0,7.134,1133,2283,7.12,138,2405,true,true,94755,0,0,0,1,1,1,138,3,5,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:01:09.468448,"""REMOVE""",7.12,138,94755,true,7.12,138,0,1,7.124,750,3034,7.114,69,826,true,true,94755,871,0,0,1,1,0,0,5,2,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.086,82,272540,2023-09-01 07:04:23.368240,"""INSERT""",null,0,null,false,null,0,1,0,7.096,1409,3309,7.086,82,1489,true,true,272540,0,0,0,1,1,1,82,4,4,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.086,41,272540,2023-09-01 07:04:23.395834,"""UPDATE""",7.086,82,272540,true,7.086,41,1,1,7.096,1409,3309,7.086,366,1123,false,true,272540,2443,0,0,1,2,2,366,4,3,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:04:23.404833,"""REMOVE""",7.086,41,272540,false,null,0,0,1,7.094,960,4269,5.0,757,757,true,true,272540,0,0,0,1,1,0,0,5,1,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.164,325,1473130,2023-09-01 07:29:19.004748,"""INSERT""",null,0,null,false,null,0,1,0,7.174,634,6279,7.164,325,6270,true,true,1473130,0,0,0,2,1,1,325,9,9,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.164,324,1473130,2023-09-01 07:29:20.598987,"""UPDATE""",7.164,325,1473130,true,7.164,1,1,1,7.174,325,5970,7.164,324,6159,false,true,1473130,13584,0,0,1,1,1,324,8,9,"""CONTINUOUS_TRADING""","""AQEU"""


original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,market_state_at_removal,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,i64,i64,f64,f64,list[struct[3]],str,duration[ns],f64,f64,f64
51443,"""AQEU""",2023-09-01 07:00:24.633283,2023-09-01 07:00:24.642585,2023-09-01 07:00:24.642585,0,7.104,750,0,1,7.104,8.1,[],"""CONTINUOUS_TRADING""",9302µs,7.602,0.498,0.065509
94755,"""AQEU""",2023-09-01 07:01:06.903164,2023-09-01 07:01:09.468448,2023-09-01 07:01:09.468448,0,7.12,138,138,1,7.12,7.134,[],"""CONTINUOUS_TRADING""",2s 565284µs,7.127,0.007,0.000982
272540,"""AQEU""",2023-09-01 07:04:23.368240,2023-09-01 07:04:23.404833,2023-09-01 07:04:23.404833,1,7.086,82,41,1,7.086,7.096,"[{2023-09-01 07:04:23.395834,7.086,41}]","""CONTINUOUS_TRADING""",36593µs,7.091,0.005,0.000705
1473130,"""AQEU""",2023-09-01 07:29:19.004748,2023-09-01 07:29:20.599030,2023-09-01 07:29:20.599030,1,7.164,325,1,1,7.164,7.174,"[{2023-09-01 07:29:20.598987,7.164,324}]","""CONTINUOUS_TRADING""",1s 594282µs,7.169,0.005,0.000697
